In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:95%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
.inner_cell{font-size:20pt;}
div.text_cell_render pre code {font-size:20pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:20pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:20pt;padding:5px;}
table.dataframe{font-size:20px;}
</style>
"""))

In [2]:
from tensorflow.keras.utils import to_categorical 
from tensorflow.keras.models import Sequential, load_model, save_model 
from tensorflow.keras.layers import Dense, Input 
from sklearn.preprocessing import LabelEncoder
import numpy as np
import pandas as pd 

In [9]:
df = pd.read_csv('data/ch13_apt_fillna_median.csv', encoding='utf-8')

In [8]:
# url='https://raw.githubusercontent.com/4aix/data/refs/heads/master/ch13_apt_fillna_median.csv'
# df=pd.read_csv(url)
# df.head()

,지역명,평당분양가격,연도,월
0,서울,18189.0,2013,12
1,부산,8111.0,2013,12
2,대구,8080.0,2013,12
3,인천,10204.0,2013,12
4,광주,6098.0,2013,12


- 지역명2 : 지역명필드를 라벨인코딩
- 독립변수(X) : 지역명2 , 연도 , 월
- 종속변수(y) : 평당분양가격
- 독립변수와 종속변수(reshape)의 스케일 조정
     * 정규화(MinMaxScaler) 작업후 : 지역명2m, 연도m, 월m 컬럼으로 추가(df)
     * 표준화(StandardScaler) 작업후 : 지역명2s, 연도s, 월s컬럼으로 추가(df)
 => 지역명, 연도, 월, 지역명2, 지역명2m, 연도m, 월m, 평당분양가격m, 지역명2s, 연도s, 월s컬럼, 평당분양가격s 
- 데이터프레임.to_numpy(), 데이터프레임.values, np.array(데이터프레임)등을 이용하여 데이터 프레임을 넘파이 배열로 변환

# 2. 지역명의 라벨인코딩

In [25]:
loca= df.iloc[:,0]
le = LabelEncoder()
labeling_loca= le.fit_transform(loca)
labeling_loca

array([ 8,  7,  5, ...,  3,  2, 14])

In [42]:
df_add=pd.concat([df,pd.DataFrame(labeling_loca)],axis=1)
df_add.columns = ['지역명', '평당분양가격','연도','월','지역명2']
df_add

,지역명,평당분양가격,연도,월,지역명2
0,서울,18189.0,2013,12,8
1,부산,8111.0,2013,12,7
2,대구,8080.0,2013,12,5
3,인천,10204.0,2013,12,11
4,광주,6098.0,2013,12,4
...,...,...,...,...,...
2171,전북,12058.2,2024,8,13
2172,전남,13120.8,2024,8,12
2173,경북,13827.0,2024,8,3
2174,경남,13252.8,2024,8,2


# 3. MinMaxScaling

In [49]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
x1_data=np.array(df_add.iloc[:,4]).reshape(-1,1) #독립변수1: 라벨인코딩된 지역명
scaler_x1=MinMaxScaler()
norm_scaled_x1_data=scaler_x1.fit_transform(x1_data)

x2_data=np.array(df_add.iloc[:,2]).reshape(-1,1) #독립변수2: 연도
scaler_x2=MinMaxScaler()
norm_scaled_x2_data=scaler_x2.fit_transform(x2_data)

x3_data=np.array(df_add.iloc[:,3]).reshape(-1,1) #독립변수3: 월
scaler_x3=MinMaxScaler()
norm_scaled_x3_data=scaler_x3.fit_transform(x3_data)

y_data=np.array(df_add.iloc[:,1]).reshape(-1,1)  #종속변수 : 평당분양가격
scaler_y=MinMaxScaler()
norm_scaled_y_data=scaler_y.fit_transform(y_data)

# 4. StandardScaling

In [50]:
scaler_x1= StandardScaler()
stan_scaled_x1_data = scaler_x1.fit_transform(x1_data)

scaler_x2= StandardScaler()
stan_scaled_x2_data = scaler_x2.fit_transform(x2_data)

scaler_x3= StandardScaler()
stan_scaled_x3_data = scaler_x3.fit_transform(x3_data)

scaler_y= StandardScaler()
stan_scaled_y_data = scaler_y.fit_transform(y_data)


In [67]:
x0_data=np.array(df_add.iloc[:,0]).reshape(-1,1) # 지역명 문자

In [54]:
print(np.column_stack([x0_data,    #지역명
                       x2_data,    #연도
                       x3_data,    #월
                       x1_data,    #지역명2
                       norm_scaled_x1_data, #지역명2m
                       norm_scaled_x2_data, #연도m
                       norm_scaled_x3_data, #월m
                       norm_scaled_y_data,  #평당분양가격m
                       stan_scaled_x1_data, #지역명2s
                       stan_scaled_x2_data, #연도s
                       stan_scaled_x3_data, #월s
                       stan_scaled_y_data,  #평당분양가격s                           
                      ]))

[['서울' 2013 12 ... -1.8753666106028195 1.6219602522086187
  1.1685913237870176]
 ['부산' 2013 12 ... -1.8753666106028195 1.6219602522086187
  -0.7283121597774849]
 ['대구' 2013 12 ... -1.8753666106028195 1.6219602522086187
  -0.7341470484449287]
 ...
 ['경북' 2024 8 ... 1.6641993246904376 0.4637403789438868
  0.34756602161313793]
 ['경남' 2024 8 ... 1.6641993246904376 0.4637403789438868
  0.23948882571487054]
 ['제주' 2024 8 ... 1.6641993246904376 0.4637403789438868
  2.5296073388005684]]


In [55]:
scaled_data=[norm_scaled_x1_data,norm_scaled_x2_data, norm_scaled_x3_data, norm_scaled_y_data,  
stan_scaled_x1_data, stan_scaled_x2_data, stan_scaled_x3_data, stan_scaled_y_data]

In [59]:
df_scaled_data=pd.DataFrame(np.column_stack(scaled_data))

In [69]:
df_prep=pd.concat([df_add,df_scaled_data],axis=1)

In [71]:
df_prep.columns = ['지역명', '평당분양가격','연도','월','지역명2',
                   '지역명2m','연도m','월m','평당분양가격m',
                   '지역명2s','연도s','월s','평당분양가격s']

In [72]:
df_prep

,지역명,평당분양가격,연도,월,지역명2,지역명2m,연도m,월m,평당분양가격m,지역명2s,연도s,월s,평당분양가격s
0,서울,18189.0,2013,12,8,0.5000,0.0,1.000000,0.328198,0.000000,-1.875367,1.62196,1.168591
1,부산,8111.0,2013,12,7,0.4375,0.0,1.000000,0.065274,-0.204124,-1.875367,1.62196,-0.728312
2,대구,8080.0,2013,12,5,0.3125,0.0,1.000000,0.064466,-0.612372,-1.875367,1.62196,-0.734147
3,인천,10204.0,2013,12,11,0.6875,0.0,1.000000,0.119878,0.612372,-1.875367,1.62196,-0.334363
4,광주,6098.0,2013,12,4,0.2500,0.0,1.000000,0.012757,-0.816497,-1.875367,1.62196,-1.107203
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2171,전북,12058.2,2024,8,13,0.8125,1.0,0.636364,0.168252,1.020621,1.664199,0.46374,0.014639
2172,전남,13120.8,2024,8,12,0.7500,1.0,0.636364,0.195974,0.816497,1.664199,0.46374,0.214643
2173,경북,13827.0,2024,8,3,0.1875,1.0,0.636364,0.214398,-1.020621,1.664199,0.46374,0.347566
2174,경남,13252.8,2024,8,2,0.1250,1.0,0.636364,0.199418,-1.224745,1.664199,0.46374,0.239489


# 5. 지역명의 원핫인코딩

In [73]:
from tensorflow.keras.utils import to_categorical
import pandas as pd 

In [75]:
onehot_encoding_data=to_categorical(labeling_loca)
onehot_encoding_data

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 0., ..., 1., 0., 0.]], dtype=float32)